In [1]:
## Import the modules needed
import pandas as pd
import altair as alt

In [3]:
# Load data and stuff

df = pd.read_csv("PCOS_data.csv")
df.columns = df.columns.str.strip()

# renaming columns
df = df.rename(columns={
    "PCOS (Y/N)": "PCOS",
    "Weight gain(Y/N)": "Weight gain",
    "hair growth(Y/N)": "Hair growth",
    "Skin darkening (Y/N)": "Skin darkening",
    "Hair loss(Y/N)": "Hair loss",
    "Pimples(Y/N)": "Pimples",
    "Fast food (Y/N)": "Fast food",
    "Reg.Exercise(Y/N)": "Regular exercise",
    "RBS(mg/dl)": "RBS"
})

In [5]:
df.head()

,PCOS,BMI,Weight gain,Hair growth,Skin darkening,Hair loss,Pimples,Fast food,Regular exercise,Waist:Hip Ratio,RBS
0,0,19.3,0,0,0,0,0,1,0,0.83,92.0
1,0,24.9,0,0,0,0,0,0,0,0.84,92.0
2,1,25.3,0,0,0,1,1,1,0,0.90,84.0
3,0,29.7,0,0,0,0,0,0,0,0.86,76.0
4,0,20.1,0,0,0,1,0,0,0,0.81,84.0


In [4]:
# Change all the data to numbers just in case
df = df.apply(pd.to_numeric, errors="coerce")


In [6]:
df.head()

,PCOS,BMI,Weight gain,Hair growth,Skin darkening,Hair loss,Pimples,Fast food,Regular exercise,Waist:Hip Ratio,RBS
0,0,19.3,0,0,0,0,0,1,0,0.83,92.0
1,0,24.9,0,0,0,0,0,0,0,0.84,92.0
2,1,25.3,0,0,0,1,1,1,0,0.90,84.0
3,0,29.7,0,0,0,0,0,0,0,0.86,76.0
4,0,20.1,0,0,0,1,0,0,0,0.81,84.0


In [7]:
# Creating all the medians for BMI, RBS and Waist:Hip and assigning 1 if patients stats are higher and 0 if 
# patients stats are lower

#BMI
df["Higher BMI"] = (df["BMI"] > df["BMI"].median()).astype(int)

#WAIST TO HIP
df["Higher Waist:Hip Ratio"] = (df["Waist:Hip Ratio"] > df["Waist:Hip Ratio"].median()).astype(int)

#RBS
df["Higher RBS"] = (df["RBS"] > df["RBS"].median()).astype(int)

In [8]:
df.head()

,PCOS,BMI,Weight gain,Hair growth,Skin darkening,Hair loss,Pimples,Fast food,Regular exercise,Waist:Hip Ratio,RBS,Higher BMI,Higher Waist:Hip Ratio,Higher RBS
0,0,19.3,0,0,0,0,0,1,0,0.83,92.0,0,0,0
1,0,24.9,0,0,0,0,0,0,0,0.84,92.0,1,0,0
2,1,25.3,0,0,0,1,1,1,0,0.90,84.0,1,1,0
3,0,29.7,0,0,0,0,0,0,0,0.86,76.0,1,0,0
4,0,20.1,0,0,0,1,0,0,0,0.81,84.0,0,0,0


In [ ]:
# Seperating the variables for symptoms and lifestyle

symptom_vars = ["Weight gain", "Hair growth", "Skin darkening", "Hair loss", "Pimples"]
lifestyle_vars = ["Fast food", "Regular exercise", "Higher BMI", "Higher Waist:Hip Ratio", "Higher RBS"]
all_vars = symptom_vars + lifestyle_vars

In [ ]:
# Data to show when I hover
# I'm hoping that the user can hover over the bar and get more info on things like the percentages, 
# the difference between PCOS and non-PCOS, all things that MIGHT help with the task imo

def make_summary(variable_list, panel_name):
    rows = []
    for indicator in variable_list:
        for pcos_value in [0, 1]:
            group = df[df["PCOS"] == pcos_value]
            rows.append({
                "Panel": panel_name,
                "Indicator": indicator,
                "PCOS": pcos_value,
                "PCOS Label": "PCOS-positive" if pcos_value == 1 else "non-PCOS",
                "Percent": group[indicator].mean() * 100,
                "Count": group[indicator].sum(),
                "Total": group[indicator].count()
            })
    return pd.DataFrame(rows)


summary = pd.concat([
    make_summary(symptom_vars, "Symptoms"),
    make_summary(lifestyle_vars, "Lifestyle")
], ignore_index=True)

In [12]:
# Adding a difference label
# Essentially its the difference between the percentage of women with certain stats with PCOS and without

differences = []
for indicator in all_vars:
    pcos_positive = summary[(summary["Indicator"] == indicator) & (summary["PCOS"] == 1)]["Percent"].values[0]
    non_pcos = summary[(summary["Indicator"] == indicator) & (summary["PCOS"] == 0)]["Percent"].values[0]
    differences.append({
        "Indicator": indicator,
        "Difference": abs(pcos_positive - non_pcos),
        "PCOS_Positive_Percent": pcos_positive,
        "Non_PCOS_Percent": non_pcos
    })

summary = summary.merge(pd.DataFrame(differences), on="Indicator", how="left")
summary["Difference Label"] = summary["Difference"].round(1).astype(str)

In [13]:
# Creating ranks for an easier sort algorithm since i've been having so much trouble with it

rank_table = summary.drop_duplicates(["Panel", "Indicator"])[
    ["Panel", "Indicator", "Difference", "PCOS_Positive_Percent", "Non_PCOS_Percent"]
].copy()
rank_table["Rank_Difference"] = rank_table.groupby("Panel")["Difference"].rank(method="first", ascending=False)
rank_table["Rank_PCOS_Positive"] = rank_table.groupby("Panel")["PCOS_Positive_Percent"].rank(method="first", ascending=False)
rank_table["Rank_Non_PCOS"] = rank_table.groupby("Panel")["Non_PCOS_Percent"].rank(method="first", ascending=False)

summary = summary.merge(
    rank_table[["Panel", "Indicator", "Rank_Difference", "Rank_PCOS_Positive", "Rank_Non_PCOS"]],
    on=["Panel", "Indicator"], how="left"
)

In [ ]:
# Sorting interaction

sort_by = alt.param(
    name="SortBy",
    bind=alt.binding_select(
        options=["Largest difference", "Highest PCOS-positive %", "Highest non-PCOS %"],
        name="Sort by: "
    ),
    value="Largest difference"
)


# CREATING A SORT KEY
SORT_EXPR = (
    "SortBy == 'Largest difference' ? datum.Rank_Difference : "
    "SortBy == 'Highest PCOS-positive %' ? datum.Rank_PCOS_Positive : "
    "datum.Rank_Non_PCOS"
)


In [15]:
# Highlighting interaction

symptom_select = alt.selection_point(name="SymptomSelect", fields=["Indicator"], on="click", clear="dblclick", empty=True)
lifestyle_select = alt.selection_point(name="LifestyleSelect", fields=["Indicator"], on="click", clear="dblclick", empty=True)
detail_select = alt.selection_point(name="DetailSelect", fields=["Indicator"], on="click", clear="dblclick", empty=False)


In [17]:
# Creating a bar chart function that'll make it easier. you will see

def make_bar_chart(data, title, select_param, enable_detail_click=False, show_legend=False):
    diff_data = data.drop_duplicates("Indicator").copy()

    base = alt.Chart(data).transform_calculate(SortValue=SORT_EXPR)
    diff_base = alt.Chart(diff_data).transform_calculate(SortValue=SORT_EXPR)

    y_sort = alt.EncodingSortField(field="SortValue", op="min", order="ascending")
    y_enc = alt.Y("Indicator:N", sort=y_sort, title=None)
    y_offset = alt.YOffset("PCOS Label:N", scale=alt.Scale(paddingInner=0.25))

    color_enc = alt.Color(
        "PCOS Label:N",
        title="Diagnosis group",
        legend=alt.Legend(orient="top", direction="horizontal", titleFontSize=13, labelFontSize=12, symbolType="square", symbolSize=200)
        if show_legend else None,
        scale=alt.Scale(domain=["PCOS-positive", "non-PCOS"], range=["#4c78a8", "#f58518"])
    )
    bars = base.mark_bar().encode(
        y=y_enc,
        x=alt.X("Percent:Q", title="Percent of patients with indicator", scale=alt.Scale(domain=[0, 110])),
        yOffset=y_offset,
        color=color_enc,
        opacity=alt.condition(select_param, alt.value(1), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("Indicator:N", title="Indicator"),
            alt.Tooltip("PCOS Label:N", title="Diagnosis group"),
            alt.Tooltip("Percent:Q", title="Percent", format=".1f"),
            alt.Tooltip("Count:Q", title="Count", format=".0f"),
            alt.Tooltip("Total:Q", title="Group total", format=".0f"),
            alt.Tooltip("Difference:Q", title="Difference", format=".1f")
        ]
    ).add_params(select_param)

    if enable_detail_click:
        bars = bars.add_params(detail_select)

    percent_text = base.mark_text(align="left", baseline="middle", dx=4, fontSize=11).encode(
        y=y_enc, x="Percent:Q", yOffset=y_offset,
        text=alt.Text("Percent:Q", format=".0f"),
        opacity=alt.condition(select_param, alt.value(1), alt.value(0.35))
    )

    diff_text = diff_base.mark_text(align="left", baseline="middle", fontSize=11, fontWeight="bold").encode(
        y=y_enc, x=alt.value(430), text="Difference Label:N",
        tooltip=[
            alt.Tooltip("Indicator:N", title="Indicator"),
            alt.Tooltip("Difference:Q", title="Difference", format=".1f")
        ],
        opacity=alt.condition(select_param, alt.value(1), alt.value(0.35))
    )

    chart = bars + percent_text + diff_text

    if enable_detail_click:
        hint = diff_data[diff_data["Indicator"].isin(["Higher BMI", "Higher Waist:Hip Ratio", "Higher RBS"])].copy()
        hint["Hint"] = "click bar for details"
        detail_hint = alt.Chart(hint).transform_calculate(SortValue=SORT_EXPR).mark_text(
            align="left", baseline="middle", fontSize=10, fontStyle="italic"
        ).encode(
            y=y_enc, x=alt.value(485), text="Hint:N",
            tooltip=[alt.Tooltip("Indicator:N", title="Click this row to show full distribution")]
        )
        chart = chart + detail_hint

    return chart.properties(title=title, width=540, height=270)

In [18]:
# making bar charts is now easy!!!

symptom_chart = make_bar_chart(
    summary[summary["Panel"] == "Symptoms"],
    "Panel 1: Symptom indicators", symptom_select, enable_detail_click=False, show_legend=True
)
lifestyle_chart = make_bar_chart(
    summary[summary["Panel"] == "Lifestyle"],
    "Panel 2: Lifestyle", lifestyle_select, enable_detail_click=True
)

overview = alt.hconcat(symptom_chart, lifestyle_chart, spacing=70).add_params(sort_by)

In [19]:
# Making the details chart

detail_map = {"Higher BMI": "BMI", "Higher Waist:Hip Ratio": "Waist:Hip Ratio", "Higher RBS": "RBS"}
detail_rows = []
for indicator_name, measure_name in detail_map.items():
    for _, row in df.iterrows():
        detail_rows.append({
            "Indicator": indicator_name,
            "Measure": measure_name,
            "PCOS": row["PCOS"],
            "PCOS Label": "PCOS-positive" if row["PCOS"] == 1 else "non-PCOS",
            "Value": row[measure_name]
        })
detail_df = pd.DataFrame(detail_rows).dropna()

detail_base = alt.Chart(detail_df).transform_filter(detail_select)

box = detail_base.mark_boxplot(size=55).encode(
    x=alt.X("PCOS Label:N", title="Diagnosis group"),
    y=alt.Y("Value:Q", title="Original numeric value"),
    color=alt.Color("PCOS Label:N", legend=None),
    tooltip=[
        alt.Tooltip("Measure:N", title="Measure"),
        alt.Tooltip("PCOS Label:N", title="Diagnosis group"),
        alt.Tooltip("Value:Q", title="Value", format=".2f")
    ]
)
points = detail_base.mark_circle(size=32, opacity=0.30).encode(
    x=alt.X("PCOS Label:N", title="Diagnosis group"),
    y=alt.Y("Value:Q", title="Original numeric value"),
    color=alt.Color("PCOS Label:N", legend=None),
    tooltip=[
        alt.Tooltip("Measure:N", title="Measure"),
        alt.Tooltip("PCOS Label:N", title="Diagnosis group"),
        alt.Tooltip("Value:Q", title="Value", format=".2f")
    ]
)
detail_chart = (box + points).properties(
    title="Detail view: click Higher BMI, Higher Waist:Hip Ratio, or Higher RBS in Panel 2",
    width=650, height=300
)

In [20]:
# CREATING DASHBOARD PLEASE WORK

dashboard = alt.vconcat(overview, detail_chart, spacing=35).properties(
    title="PCOS Indicator Comparison Dashboard"
).configure_title(
    fontSize=18, anchor="start"
).configure_axis(
    labelFontSize=12, titleFontSize=13
).configure_legend(
    labelFontSize=12, titleFontSize=13
).configure_view(
    stroke=None
)

dashboard

alt.VConcatChart(...)

In [21]:
dashboard.save("pcos_dashboard_visualization.html")